<a href="https://colab.research.google.com/github/shamikkarkhanis/csci4170-paiml/blob/main/hw5_part_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

In [ ]:
def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(query, key, value, mask=None):
    query = np.asarray(query, dtype=float)
    key = np.asarray(key, dtype=float)
    value = np.asarray(value, dtype=float)

    d_k = key.shape[-1]
    scores = query @ key.T
    scaled_scores = scores / np.sqrt(d_k)

    if mask is not None:
        scaled_scores = np.where(mask, scaled_scores, -1e9)

    attention_weights = softmax(scaled_scores, axis=-1)
    output = attention_weights @ value

    return output, attention_weights, scaled_scores


In [ ]:
tokens = ["I", "want", "apples"]

Q = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 1.0],
])

K = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0],
])

V = np.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [1.0, 1.0],
])

attention_output, attention_weights, scaled_scores = scaled_dot_product_attention(Q, K, V)

print("Q:")
display(pd.DataFrame(Q, index=tokens, columns=["q1", "q2", "q3", "q4"]))

print("K:")
display(pd.DataFrame(K, index=tokens, columns=["k1", "k2", "k3", "k4"]))

print("V:")
display(pd.DataFrame(V, index=tokens, columns=["v1", "v2"]))

print("QK^T / sqrt(d_k)")
display(pd.DataFrame(scaled_scores, index=tokens, columns=tokens))

print("weights after softmax:")
display(pd.DataFrame(attention_weights, index=tokens, columns=tokens))

print("attention output:")
display(pd.DataFrame(attention_output, index=tokens, columns=["out_1", "out_2"]))


Q:


,q1,q2,q3,q4
I,1.0,0.0,1.0,0.0
want,0.0,2.0,0.0,1.0
apples,1.0,1.0,0.0,1.0


K:


,k1,k2,k3,k4
I,1.0,0.0,1.0,0.0
want,0.0,1.0,0.0,1.0
apples,1.0,1.0,0.0,0.0


V:


...

,I,want,apples
I,1.0,0.0,0.5
want,0.0,1.5,1.0
apples,0.5,1.0,1.0


weights after softmax:


,I,want,apples
I,0.506480,0.186324,0.307196
want,0.121952,0.546549,0.331499
apples,0.232697,0.383652,0.383652


attention output:


,out_1,out_2
I,0.813676,0.679843
want,0.453451,1.424598
apples,0.616348,1.150955


In [ ]:
causal_mask = np.tril(np.ones((len(tokens), len(tokens)), dtype=bool))

masked_output, masked_weights, masked_scores = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Causal mask:")
display(pd.DataFrame(causal_mask.astype(int), index=tokens, columns=tokens))

print("scaled scores:")
display(pd.DataFrame(masked_scores, index=tokens, columns=tokens))

print("attention weights:")
display(pd.DataFrame(masked_weights, index=tokens, columns=tokens))

print("attention output:")
display(pd.DataFrame(masked_output, index=tokens, columns=["out_1", "out_2"]))


Causal mask:


,I,want,apples
I,1,0,0
want,1,1,0
apples,1,1,1


scaled scores:


,I,want,apples
I,1.0,-1.000000e+09,-1.000000e+09
want,0.0,1.500000e+00,-1.000000e+09
apples,0.5,1.000000e+00,1.000000e+00


attention weights:


,I,want,apples
I,1.000000,0.000000,0.000000
want,0.182426,0.817574,0.000000
apples,0.232697,0.383652,0.383652


attention output:


,out_1,out_2
I,1.000000,0.000000
want,0.182426,1.635149
apples,0.616348,1.150955


In [ ]:
row_sums = attention_weights.sum(axis=1)
masked_row_sums = masked_weights.sum(axis=1)

print("Row sums of unmasked attention weights:", row_sums)
print("Row sums of masked attention weights:", masked_row_sums)


Row sums of unmasked attention weights: [1. 1. 1.]
Row sums of masked attention weights: [1. 1. 1.]


## Part 2

In [ ]:
def tanh(x):
    return np.tanh(x)

In [ ]:
# toy data
source_tokens = ["<bos>", "i", "like", "apples", "<eos>"]
target_tokens = ["<bos>", "j", "aime", "pommes", "<eos>"]

src_vocab = {token: idx for idx, token in enumerate(sorted(set(source_tokens)))}
tgt_vocab = {token: idx for idx, token in enumerate(sorted(set(target_tokens)))}
tgt_id_to_token = {idx: token for token, idx in tgt_vocab.items()}

src_ids = np.array([src_vocab[token] for token in source_tokens])
tgt_ids = np.array([tgt_vocab[token] for token in target_tokens])

pd.DataFrame({
    "source_token": source_tokens,
    "source_id": src_ids,
    "target_token": target_tokens,
    "target_id": tgt_ids,
})


,source_token,source_id,target_token,target_id
0,<bos>,0,<bos>,0
1,i,3,j,3
2,like,4,aime,2
3,apples,2,pommes,4
4,<eos>,1,<eos>,1


In [ ]:
embed_dim = 6
hidden_dim = 8

src_embedding = rng.normal(0, 0.5, size=(len(src_vocab), embed_dim))
tgt_embedding = rng.normal(0, 0.5, size=(len(tgt_vocab), embed_dim))

W_xh_enc = rng.normal(0, 0.4, size=(embed_dim, hidden_dim))
W_hh_enc = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
b_enc = np.zeros(hidden_dim)

W_xh_dec = rng.normal(0, 0.4, size=(embed_dim, hidden_dim))
W_hh_dec = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
W_ctx_dec = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
b_dec = np.zeros(hidden_dim)

W_out = rng.normal(0, 0.4, size=(hidden_dim * 2, len(tgt_vocab)))
b_out = np.zeros(len(tgt_vocab))


In [ ]:
# using rnn to create hidden states

def rnn_encoder_forward(token_ids, embedding_matrix, W_xh, W_hh, b):
    hidden_states = []
    h_t = np.zeros(W_hh.shape[0])

    for token_id in token_ids:
        x_t = embedding_matrix[token_id]
        h_t = tanh(x_t @ W_xh + h_t @ W_hh + b)
        hidden_states.append(h_t.copy())

    return np.vstack(hidden_states)


encoder_states = rnn_encoder_forward(src_ids, src_embedding, W_xh_enc, W_hh_enc, b_enc)
encoder_memory, encoder_self_attention, encoder_scores = scaled_dot_product_attention(
    encoder_states,
    encoder_states,
    encoder_states,
)

print("Raw encoder hidden states shape:", encoder_states.shape)
print("Attention-enhanced encoder memory shape:", encoder_memory.shape)


Raw encoder hidden states shape: (5, 8)
Attention-enhanced encoder memory shape: (5, 8)


In [ ]:
encoder_state_df = pd.DataFrame(
    encoder_states,
    index=source_tokens,
    columns=[f"h_{i}" for i in range(hidden_dim)],
)

encoder_memory_df = pd.DataFrame(
    encoder_memory,
    index=source_tokens,
    columns=[f"m_{i}" for i in range(hidden_dim)],
)

encoder_attention_df = pd.DataFrame(
    encoder_self_attention,
    index=[f"query:{token}" for token in source_tokens],
    columns=[f"key:{token}" for token in source_tokens],
)

print("Encoder hidden states:")
display(encoder_state_df)

print("Encoder self-attention weights:")
display(encoder_attention_df)

print("Attention-enhanced encoder memory:")
display(encoder_memory_df)


Encoder hidden states:


,h_0,h_1,h_2,h_3,h_4,h_5,h_6,h_7
<bos>,-0.269416,-0.249463,0.346875,-0.350698,-0.116391,0.124554,0.117498,-0.746079
i,-0.513908,-0.813693,-0.495479,0.266450,0.417579,0.402838,0.493522,0.163108
like,-0.481125,0.683042,0.535469,-0.074562,-0.317823,0.330381,0.124946,0.507960
apples,0.028259,-0.491275,-0.527727,-0.030743,-0.159856,0.032641,-0.872996,0.848073
<eos>,-0.106191,0.923947,0.808054,0.480003,-0.793054,0.217725,0.156947,-0.871782


Encoder self-attention weights:


,key:<bos>,key:i,key:like,key:apples,key:<eos>
query:<bos>,0.260695,0.185426,0.177157,0.140539,0.236184
query:i,0.181839,0.347956,0.154392,0.200648,0.115165
query:like,0.164744,0.146406,0.288947,0.157282,0.242620
query:apples,0.141143,0.205485,0.169859,0.379765,0.103748
query:<eos>,0.182852,0.090918,0.201986,0.079977,0.444266


Attention-enhanced encoder memory:


,m_0,m_1,m_2,m_3,m_4,m_5,m_6,m_7
<bos>,-0.271871,0.054271,0.210099,0.053821,-0.218990,0.221707,0.058657,-0.160980
i,-0.308649,-0.215203,-0.039485,0.066542,-0.048342,0.245451,0.055290,0.069278
like,-0.279963,0.184035,0.252375,0.071313,-0.267426,0.232919,0.028486,-0.030383
apples,-0.225635,-0.177102,-0.078479,0.030712,-0.127593,0.191460,-0.176032,0.246117
<eos>,-0.238084,0.389558,0.443321,0.155829,-0.412625,0.225471,0.091499,-0.338469


In [ ]:
def decoder_forward(target_ids, embedding_matrix, encoder_memory, W_xh, W_hh, W_ctx, b, W_out, b_out):
    h_t = np.zeros(W_hh.shape[0])
    contexts = []
    attn_weights_all = []
    logits_all = []

    for token_id in target_ids[:-1]:
        x_t = embedding_matrix[token_id]
        context, attn_weights, _ = scaled_dot_product_attention(
            h_t.reshape(1, -1),
            encoder_memory,
            encoder_memory,
        )
        context = context[0]
        h_t = tanh(x_t @ W_xh + h_t @ W_hh + context @ W_ctx + b)

        combined = np.concatenate([h_t, context])
        logits = combined @ W_out + b_out

        contexts.append(context.copy())
        attn_weights_all.append(attn_weights[0].copy())
        logits_all.append(logits.copy())

    return np.vstack(logits_all), np.vstack(attn_weights_all), np.vstack(contexts)


decoder_logits, decoder_attention, decoder_contexts = decoder_forward(
    tgt_ids,
    tgt_embedding,
    encoder_memory,
    W_xh_dec,
    W_hh_dec,
    W_ctx_dec,
    b_dec,
    W_out,
    b_out,
)

decoder_probabilities = softmax(decoder_logits, axis=1)
predicted_ids = decoder_probabilities.argmax(axis=1)
predicted_tokens = [tgt_id_to_token[idx] for idx in predicted_ids]

print("Decoder logits shape:", decoder_logits.shape)
print("Decoder attention shape:", decoder_attention.shape)
print("Predicted target tokens:", predicted_tokens)


Decoder logits shape: (4, 5)
Decoder attention shape: (4, 5)
Predicted target tokens: ['aime', 'aime', '<eos>', '<bos>']


In [ ]:
decoder_steps = [f"step_{i+1}" for i in range(len(target_tokens) - 1)]

decoder_attention_df = pd.DataFrame(
    decoder_attention,
    index=decoder_steps,
    columns=source_tokens,
)

decoder_context_df = pd.DataFrame(
    decoder_contexts,
    index=decoder_steps,
    columns=[f"c_{i}" for i in range(hidden_dim)],
)

decoder_probs_df = pd.DataFrame(
    decoder_probabilities,
    index=decoder_steps,
    columns=[tgt_id_to_token[i] for i in range(len(tgt_vocab))],
)

print("Decoder attention over encoder memory:")
display(decoder_attention_df)

print("Decoder context vectors:")
display(decoder_context_df)

print("Decoder output probabilities:")
display(decoder_probs_df)


Decoder attention over encoder memory:


,<bos>,i,like,apples,<eos>
step_1,0.200000,0.200000,0.200000,0.200000,0.200000
step_2,0.207232,0.190505,0.205346,0.177915,0.219003
step_3,0.201207,0.207590,0.199562,0.194656,0.196987
step_4,0.198059,0.176563,0.208741,0.189315,0.227323


Decoder context vectors:


,c_0,c_1,c_2,c_3,c_4,c_5,c_6,c_7
step_1,-0.264841,0.047112,0.157566,0.075643,-0.214995,0.223401,0.011580,-0.042888
step_2,-0.264914,0.061846,0.170967,0.078065,-0.222572,0.223975,0.017258,-0.056739
step_3,-0.265465,0.045236,0.156493,0.075548,-0.213584,0.223727,0.012723,-0.042838
step_4,-0.263620,0.066195,0.173241,0.078532,-0.225685,0.223369,0.014800,-0.056342


Decoder output probabilities:


,<bos>,<eos>,aime,j,pommes
step_1,0.067781,0.238939,0.299606,0.173683,0.219991
step_2,0.129371,0.139137,0.261860,0.236505,0.233127
step_3,0.100316,0.310612,0.290169,0.177329,0.121575
step_4,0.251599,0.200267,0.204964,0.206240,0.136929


## Part 3

In [ ]:
!pip install datasets sacrebleu

from datasets import load_dataset
from collections import Counter
import sacrebleu


In [ ]:
dataset = load_dataset("bentrevett/multi30k")

train_data = dataset['train']
test_data = dataset['test']

subset_size = 1000
train_source = [example['de'] for example in train_data][:subset_size]
train_target = [example['en'] for example in train_data][:subset_size]

source_sentences_preprocessed = ["<bos> " + s.lower() + " <eos>" for s in train_source]
target_sentences_preprocessed = ["<bos> " + t.lower() + " <eos>" for t in train_target]

print("Source Sentences (preprocessed):")
print(source_sentences_preprocessed[:2])
print("Target Sentences (preprocessed):")
print(target_sentences_preprocessed[:2])

Source Sentences (preprocessed):
['<bos> zwei junge weiße männer sind im freien in der nähe vieler büsche. <eos>', '<bos> mehrere männer mit schutzhelmen bedienen ein antriebsradsystem. <eos>']
Target Sentences (preprocessed):
['<bos> two young, white males are outside near many bushes. <eos>', '<bos> several men in hard hats are operating a giant pulley system. <eos>']


In [ ]:

def tokenize(text):
    return text.split()

min_freq = 2

src_counter = Counter(token for s in source_sentences_preprocessed for token in tokenize(s))
tgt_counter = Counter(token for t in target_sentences_preprocessed for token in tokenize(t))

src_vocab_mt = {token: idx + 2 for idx, (token, count) in enumerate(src_counter.items()) if count >= min_freq}
src_vocab_mt["<unk>"] = 0
src_vocab_mt["<pad>"] = 1

tgt_vocab_mt = {token: idx + 2 for idx, (token, count) in enumerate(tgt_counter.items()) if count >= min_freq}
tgt_vocab_mt["<unk>"] = 0
tgt_vocab_mt["<pad>"] = 1

tgt_id_to_token_mt = {idx: token for token, idx in tgt_vocab_mt.items()}

def sentence_to_ids(sentence, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokenize(sentence)]

src_ids_mt = [sentence_to_ids(s, src_vocab_mt) for s in source_sentences_preprocessed]
tgt_ids_mt = [sentence_to_ids(t, tgt_vocab_mt) for t in target_sentences_preprocessed]

print(f"Source Vocabulary Size: {len(src_vocab_mt)}")
print(f"Target Vocabulary Size: {len(tgt_vocab_mt)}")
print("Numerical Source IDs for first sentence:", src_ids_mt[0])
print("Numerical Target IDs for first sentence:", tgt_ids_mt[0])

Source Vocabulary Size: 841
Target Vocabulary Size: 864
Numerical Source IDs for first sentence: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 0, 15]
Numerical Target IDs for first sentence: [2, 3, 0, 5, 6, 7, 8, 9, 10, 11, 12]
